In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
cd /content/drive/MyDrive/247/emg2qwerty_FinalProject-main

/content/drive/MyDrive/247/emg2qwerty_FinalProject-main


In [3]:
!pip install -r requirements.txt

  Using cached https://github.com/kpu/kenlm/archive/master.zip
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [4]:
import glob
from pathlib import Path
DATA_DIR = Path('Data')

all_sessions = sorted(DATA_DIR.rglob('*.hdf5'))
if not all_sessions:
    raise FileNotFoundError(f'No HDF5 files found under {DATA_DIR}')
print(f'Found {len(all_sessions)} session(s):', [s.name for s in all_sessions])

n = len(all_sessions)
train_sessions = all_sessions[: int(n * 0.70)]
val_sessions   = all_sessions[int(n * 0.70) : int(n * 0.85)]
test_sessions  = all_sessions[int(n * 0.85) :]

if not train_sessions: train_sessions = all_sessions
if not val_sessions:   val_sessions   = all_sessions
if not test_sessions:  test_sessions  = all_sessions

WINDOW_LENGTH = 600
PADDING = (150, 150)
BATCH_SIZE = 32
NUM_WORKERS = 2
# model
IN_FEATURES = 528
MLP_FEATURES = [384, 384]
HIDDEN_SIZE = 256
NUM_LAYERS = 3
RNN_TYPE = 'LSTM'
DROPOUT = 0.4
BIDIRECTIONAL = True
#training
MAX_EPOCHS = 50
LEARNING_RATE = 5e-4
WEIGHT_DECAY = 1e-5

Found 18 session(s): ['2021-06-02-1622679967-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622681518-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-02-1622682789-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622764398-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622765527-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-03-1622766673-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622861066-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622862148-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-04-1622863166-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-05-1622884635-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b4f5f.hdf5', '2021-06-05-1622885888-keystrokes-dca-study@1-0efbe614-9ae6-4131-9192-4398359b

In [5]:
from collections.abc import Sequence
from typing import Any, ClassVar

import numpy as np
import pytorch_lightning as pl
import torch
from torch import nn
from torchmetrics import MetricCollection

from emg2qwerty.charset import charset
from emg2qwerty.data import LabelData
from emg2qwerty.metrics import CharacterErrorRates
from emg2qwerty.modules import (
    MultiBandRotationInvariantMLP,
    RNNEncoder,
    SpectrogramNorm,
)


class RNNCTCModule(pl.LightningModule):

    NUM_BANDS:         ClassVar[int] = 2
    ELECTRODE_CHANNELS: ClassVar[int] = 16

    def __init__(
        self,
        in_features:   int,
        mlp_features:  Sequence[int],
        hidden_size:   int   = 256,
        num_layers:    int   = 3,
        rnn_type:      str   = 'LSTM',
        dropout:       float = 0.2,
        bidirectional: bool  = True,
        lr:            float = 5e-4,
        weight_decay:  float = 1e-5,
    ) -> None:
        super().__init__()
        self.save_hyperparameters()

        num_features = self.NUM_BANDS * mlp_features[-1]

        self.model = nn.Sequential(
            SpectrogramNorm(channels=self.NUM_BANDS * self.ELECTRODE_CHANNELS),
            MultiBandRotationInvariantMLP(
                in_features=in_features,
                mlp_features=mlp_features,
                num_bands=self.NUM_BANDS,
            ),
            nn.Flatten(start_dim=2),
            RNNEncoder(
                num_features=num_features,
                hidden_size=hidden_size,
                num_layers=num_layers,
                rnn_type=rnn_type,
                dropout=dropout,
                bidirectional=bidirectional,
            ),
            nn.Linear(num_features, charset().num_classes),
            nn.LogSoftmax(dim=-1),
        )

        self.ctc_loss = nn.CTCLoss(blank=charset().null_class)

        metrics = MetricCollection([CharacterErrorRates()])
        self.metrics = nn.ModuleDict({
            f'{phase}_metrics': metrics.clone(prefix=f'{phase}/')
            for phase in ['train', 'val', 'test']
        })
    def forward(self, inputs: torch.Tensor) -> torch.Tensor:
        return self.model(inputs)
    def _step(self, phase: str, batch: dict) -> torch.Tensor:
        inputs         = batch['inputs']
        targets        = batch['targets']
        input_lengths  = batch['input_lengths']
        target_lengths = batch['target_lengths']
        N = len(input_lengths)

        emissions = self.forward(inputs)
        T_diff = inputs.shape[0] - emissions.shape[0]
        emission_lengths = input_lengths - T_diff

        loss = self.ctc_loss(
            log_probs=emissions,
            targets=targets.transpose(0, 1),
            input_lengths=emission_lengths,
            target_lengths=target_lengths,
        )
        greedy = emissions.argmax(dim=-1)  # (T, N)
        metrics = self.metrics[f'{phase}_metrics']
        targets_np        = targets.detach().cpu().numpy()
        target_lengths_np = target_lengths.detach().cpu().numpy()
        greedy_np         = greedy.detach().cpu().numpy()

        for i in range(N):
            seq = greedy_np[:, i].tolist()
            decoded = []
            prev = None
            for c in seq:
                if c != charset().null_class and c != prev:
                    decoded.append(c)
                prev = c
            pred   = LabelData.from_labels(decoded)
            target = LabelData.from_labels(targets_np[: target_lengths_np[i], i])
            metrics.update(prediction=pred, target=target)

        self.log(f'{phase}/loss', loss, batch_size=N, sync_dist=True, prog_bar=True)
        return loss

    def _epoch_end(self, phase: str) -> None:
        metrics = self.metrics[f'{phase}_metrics']
        self.log_dict(metrics.compute(), sync_dist=True)
        metrics.reset()

    def training_step(self,   batch, batch_idx): return self._step('train', batch)
    def validation_step(self, batch, batch_idx): return self._step('val',   batch)
    def test_step(self,       batch, batch_idx): return self._step('test',  batch)

    def on_train_epoch_end(self):      self._epoch_end('train')
    def on_validation_epoch_end(self): self._epoch_end('val')
    def on_test_epoch_end(self):       self._epoch_end('test')
    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(
            self.parameters(),
            lr=self.hparams.lr,
            weight_decay=self.hparams.weight_decay,
        )
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=MAX_EPOCHS, eta_min=1e-6
        )
        return {'optimizer': optimizer, 'lr_scheduler': scheduler}


model = RNNCTCModule(
    in_features=IN_FEATURES,
    mlp_features=MLP_FEATURES,
    hidden_size=HIDDEN_SIZE,
    num_layers=NUM_LAYERS,
    rnn_type=RNN_TYPE,
    dropout=DROPOUT,
    bidirectional=BIDIRECTIONAL,
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
)

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Model ready — {total_params:,} trainable parameters')

Model ready — 6,428,835 trainable parameters


In [6]:
from torch.utils.data import ConcatDataset, DataLoader
from emg2qwerty.data import WindowedEMGDataset
from emg2qwerty.transforms import Compose, ToTensor, LogSpectrogram, RandomBandRotation, SpecAugment

transform = Compose([ToTensor(), LogSpectrogram()])
train_transform = Compose([ToTensor(), LogSpectrogram(), RandomBandRotation(), SpecAugment(),])
def make_dataset(sessions, window_length, padding, jitter, transformation):
    return ConcatDataset([
        WindowedEMGDataset(
            p,
            transform=transformation,
            window_length=window_length,
            padding=padding,
            jitter=jitter,
        )
        for p in sessions
    ])
zeropad = (0,0)
train_ds = make_dataset(train_sessions, 3000, PADDING, True, train_transform)
val_ds   = make_dataset(val_sessions,   3000, PADDING, False, transform)
test_ds  = make_dataset(test_sessions,  None , zeropad , False , transform)

collate = WindowedEMGDataset.collate

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,  shuffle=True,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE,  shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=1,           shuffle=False,
                          num_workers=NUM_WORKERS, collate_fn=collate, pin_memory=True)

print(f'Train windows : {len(train_ds):,}')
print(f'Val   windows : {len(val_ds):,}')
print(f'Test  sessions: {len(test_ds):,}')

Train windows : 7,998
Val   windows : 1,772
Test  sessions: 3


In [33]:
from emg2qwerty.charset import charset

def print_sample(loader, split_name):
    batch = next(iter(loader))
    inputs         = batch['inputs']
    targets        = batch['targets']
    target_lengths = batch['target_lengths']
    input_lengths  = batch['input_lengths']

    print(f'{"="*50}')
    print(f'  {split_name} example')
    print(f'{"="*50}')
    print(f'  Batch inputs shape : {inputs.shape}')
    print(f'  (T={inputs.shape[0]}, N={inputs.shape[1]}, bands=2, electrodes=16, freq=33)')
    print()

    # Show first 3 samples in the batch
    for i in range(min(3, inputs.shape[1])):
        chars = ''.join(
            charset().label_to_char(int(targets[t, i]))
            for t in range(target_lengths[i])
        )
        print(f'  [{i}] T={input_lengths[i].item()} timesteps  '
              f'| {target_lengths[i].item()} chars | "{chars}"')
    print()

print_sample(train_loader, 'TRAIN')
print_sample(test_loader,  'TEST')

  TRAIN example
  Batch inputs shape : torch.Size([40, 32, 2, 16, 33])
  (T=40, N=32, bands=2, electrodes=16, freq=33)

  [0] T=40 timesteps  | 1 chars | "f"
  [1] T=40 timesteps  | 0 chars | ""
  [2] T=40 timesteps  | 0 chars | ""



KeyboardInterrupt: 

In [14]:
import inspect
import emg2qwerty.transforms as T
print([name for name in dir(T) if not name.startswith('_')])

['Any', 'Callable', 'Compose', 'ForEach', 'Lambda', 'LogSpectrogram', 'RandomBandRotation', 'Sequence', 'SpecAugment', 'TTransformIn', 'TTransformOut', 'TemporalAlignmentJitter', 'ToTensor', 'Transform', 'TypeVar', 'dataclass', 'np', 'torch', 'torchaudio']


In [7]:
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Callback

class PrintCER(Callback):
    def on_validation_epoch_end(self, trainer, pl_module):
        cer = trainer.callback_metrics.get('val/CER')
        loss = trainer.callback_metrics.get('val/loss')
        if cer is not None:
            print(f'  Epoch {trainer.current_epoch:>3} | val/loss: {loss:.4f} | val/CER: {cer:.4f}')

checkpoint_cb = ModelCheckpoint(
    dirpath='checkpoints/rnn',
    filename='epoch={epoch:02d}-val_cer={val/cer:.4f}',
    monitor='val/CER',
    mode='min',
    save_top_k=3,
    auto_insert_metric_name=False,
)

early_stop_cb = EarlyStopping(
    monitor='val/CER',
    patience=10,
    mode='min',
)

trainer = pl.Trainer(
    max_epochs=MAX_EPOCHS,
    accelerator='gpu',
    devices=1,
    callbacks=[checkpoint_cb, early_stop_cb, PrintCER()],
    log_every_n_steps=10,
    gradient_clip_val=1.0,
)

trainer.fit(model, train_loader, val_loader)

INFO:pytorch_lightning.utilities.rank_zero:GPU available: True (cuda), used: True
INFO:pytorch_lightning.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO:pytorch_lightning.utilities.rank_zero:IPU available: False, using: 0 IPUs
INFO:pytorch_lightning.utilities.rank_zero:HPU available: False, using: 0 HPUs
/usr/local/lib/python3.12/dist-packages/pytorch_lightning/callbacks/model_checkpoint.py:604: UserWarning: Checkpoint directory checkpoints/rnn exists and is not empty.
  rank_zero_warn(f"Checkpoint directory {dirpath} exists and is not empty.")
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.callbacks.model_summary:
  | Name     | Type       | Params
----------------------------------------
0 | model    | Sequential | 6.4 M 
1 | ctc_loss | CTCLoss    | 0     
2 | metrics  | ModuleDict | 0     
----------------------------------------
6.4 M     Trainable params
0         Non-trainable params
6.4 M     Total para

Sanity Checking: 0it [00:00, ?it/s]

Training: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

Validation: 0it [00:00, ?it/s]

...

Validation: 0it [00:00, ?it/s]

  Epoch  48 | val/loss: 0.7882 | val/CER: 18.9579


Validation: 0it [00:00, ?it/s]

  Epoch  49 | val/loss: 0.7725 | val/CER: 19.0780


INFO:pytorch_lightning.utilities.rank_zero:`Trainer.fit` stopped: `max_epochs=50` reached.


In [8]:
trainer.test(model, test_loader, ckpt_path='best')

INFO:pytorch_lightning.utilities.rank_zero:Restoring states from the checkpoint path at checkpoints/rnn/epoch=44-val_cer=0.0000.ckpt
INFO:pytorch_lightning.accelerators.cuda:LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:pytorch_lightning.utilities.rank_zero:Loaded model weights from checkpoint at checkpoints/rnn/epoch=44-val_cer=0.0000.ckpt


Testing: 0it [00:00, ?it/s]

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃   Runningstage.testing    ┃                           ┃
┃          metric           ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/CER          │    15.256922721862793     │
│         test/DER          │    1.2978193759918213     │
│         test/IER          │     2.301365613937378     │
│         test/SER          │    11.657737731933594     │
│         test/loss         │    0.6293365955352783     │
└───────────────────────────┴───────────────────────────┘

[{'test/loss': 0.6293365955352783,
  'test/CER': 15.256922721862793,
  'test/IER': 2.301365613937378,
  'test/DER': 1.2978193759918213,
  'test/SER': 11.657737731933594}]